# CorrDiff - Fase 12 - Análise Multivariada

PCA, colinearidade, modelos temporais forward e estabilidade dos coeficientes.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

OUT = Path('../analysis_outputs/12_multivariate')
summary = json.loads((OUT/'analysis_summary.json').read_text())
summary


## 1. Folds temporais

In [ ]:
folds = pd.read_parquet(OUT/'temporal_fold_definition.parquet')
display(folds)


## 2. PCA

In [ ]:
pca = pd.read_parquet(OUT/'pca_explained_variance.parquet')
plt.figure(figsize=(8,4))
plt.plot(pca.component_index, pca.cumulative_explained_variance_ratio, marker='o')
plt.axhline(0.90, linestyle='--')
plt.axhline(0.95, linestyle='--')
plt.xlabel('Número de componentes')
plt.ylabel('Variância explicada acumulada')
plt.tight_layout()
plt.show()


## 3. Loadings

In [ ]:
load = pd.read_parquet(OUT/'pca_loadings.parquet')
for pc in ['PC1','PC2','PC3','PC4','PC5']:
    t = load[load.component.eq(pc)].copy()
    t['abs_loading'] = t.loading.abs()
    display(t.sort_values('abs_loading', ascending=False).head(10)[['feature','loading']])


## 4. Métricas binárias por fold

In [ ]:
binary = pd.read_parquet(OUT/'binary_model_metrics.parquet')
display(binary.sort_values(['event_id','fold_id','brier']))


## 5. >=45 dBZ

In [ ]:
t = binary[binary.event_id.eq('ge_45')]
for (model, rep), g in t.groupby(['model','representation']):
    g = g.sort_values('fold_id')
    plt.figure(figsize=(7,4))
    plt.plot(g.fold_id, g.brier_skill_vs_climatology, marker='o')
    plt.axhline(0)
    plt.ylabel('Brier Skill vs climatologia')
    plt.title(f'{model} - {rep}')
    plt.tight_layout()
    plt.show()


## 6. Estabilidade de coeficientes

In [ ]:
stability = pd.read_parquet(OUT/'coefficient_stability.parquet')
t = stability[(stability.target_type.eq('binary')) & (stability.target.eq('ge_45')) & (stability.representation.eq('means_plus_std'))].copy()
display(t.sort_values(['sign_consistency','coefficient_abs_mean'], ascending=[False,False]).head(20))


## 7. Targets contínuos

In [ ]:
cont = pd.read_parquet(OUT/'continuous_model_metrics.parquet')
display(cont.sort_values(['target','fold_id','rmse']))


## Regra de interpretação

Priorize estabilidade temporal, skill contra climatologia e coerência entre famílias. Coeficientes individuais podem mudar devido à forte colinearidade entre predictors raw e derivados.